# Enterprise Knowledge Mining System


## Imports & Configuration


In [1]:
import os
import re
import json
import time
import gc
import logging
import threading
import xml.etree.ElementTree as ET

import dotenv
import pymupdf as fitz
import chromadb
import spacy

from openai import OpenAI, RateLimitError
from langchain_text_splitters import RecursiveCharacterTextSplitter
from huggingface_hub import hf_hub_download, HfApi, list_repo_files
from huggingface_hub.utils import disable_progress_bars

dotenv.load_dotenv()

OPENAI_KEY  = os.getenv("OPENAI_KEY")
HF_TOKEN    = os.getenv("HF_TOKEN")
HF_REPO_ID  = os.getenv("HF_REPO_ID")

client  = OpenAI(api_key=OPENAI_KEY)
hf_api  = HfApi(token=HF_TOKEN)

logging.basicConfig(filename="pipeline.log", level=logging.INFO)
print("Imports OK")

c:\Users\jm201\OneDrive\Documents\GitHub\Enterprise-Knowledge-Mining-System\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


In [ ]:
# Configuration

SEARCH_QUERY      = "all:computer"
MAX_RESULTS       = 1000       
ARXIV_PAGE_SIZE   = 100        
REQUEST_DELAY     = 3.0        

MIN_PAGE_LENGTH   = 100        
CHUNK_SIZE        = 800
CHUNK_OVERLAP     = 50
MAX_CHUNKS_PER_PAPER = 300

EMBEDDING_MODEL   = "text-embedding-3-small"
EMBED_BATCH_SIZE  = 200        
RAG_MODEL         = "gpt-4.1-mini"

CHROMA_PATH       = "./chroma_db"
COLLECTION_NAME   = "research_papers"
CHROMA_UPSERT_BATCH = 2000    

PDF_DIR           = "pdfs"
CHECKPOINT_FILE   = "checkpoint.json"
UPLOAD_BATCH_SIZE = 20

BASE_URL = "https://export.arxiv.org/api/query"
NS = {
    "atom":  "http://www.w3.org/2005/Atom",
    "arxiv": "http://arxiv.org/schemas/atom",
}

EXCLUDED_SECTIONS = {"References", "Acknowledgment", "Acknowledgements", "Acknowledgments"}

os.makedirs(PDF_DIR, exist_ok=True)
print("Config OK")

Config OK


## Hugging Face helpers


In [ ]:
# HuggingFace cache & upload queue

_hf_files_cache  = None
_hf_cache_lock   = threading.Lock()
hf_upload_queue  = []
hf_upload_lock   = threading.Lock()
_arxiv_semaphore = threading.Semaphore(4)


def get_hf_files():
    # Returns the set of filenames in the HF repo.
    global _hf_files_cache
    with _hf_cache_lock:
        if _hf_files_cache is None:
            try:
                _hf_files_cache = set(list_repo_files(
                    repo_id=HF_REPO_ID, repo_type="dataset", token=HF_TOKEN
                ))
                print(f"HF repo contains {len(_hf_files_cache)} files.")
            except Exception as e:
                print(f"HF fetch failed: {e}")
                _hf_files_cache = set()
        return _hf_files_cache


def pdf_exists_in_hf(filename):
    return filename in get_hf_files()


def upload_to_hf(local_path):
    filename = os.path.basename(local_path)
    disable_progress_bars()
    hf_api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=filename, 
        repo_id=HF_REPO_ID,
        repo_type="dataset",
        token=HF_TOKEN,
    )
    with _hf_cache_lock:
        _hf_files_cache.add(filename)


def download_from_hf(filename, local_path):
    hf_hub_download(
        repo_id=HF_REPO_ID,
        filename=filename,       
        repo_type="dataset",
        local_dir=os.path.dirname(local_path),
    )


def queue_hf_upload(local_path):
    batch = None
    with hf_upload_lock:
        hf_upload_queue.append(local_path)
        if len(hf_upload_queue) >= UPLOAD_BATCH_SIZE:
            batch = hf_upload_queue[:]
            hf_upload_queue.clear()
    if batch:
        print(f"Uploading batch of {len(batch)} files...")
        for p in batch:
            upload_to_hf(p)


def flush_hf_uploads():
    with hf_upload_lock:
        if not hf_upload_queue:
            return
        batch = hf_upload_queue[:]
        hf_upload_queue.clear()
    print(f"Uploading final batch of {len(batch)} files to HF...")
    for p in batch:
        try:
            upload_to_hf(p)
        except Exception as e:
            print(f"Upload failed for {p}: {e}")


print("HF helpers OK")

HF helpers OK


## Checkpoint


In [4]:
def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        return set(json.load(open(CHECKPOINT_FILE)))
    return set()


def save_checkpoint(ids):
    json.dump(list(ids), open(CHECKPOINT_FILE, "w"))


print("Checkpoint helpers OK")

Checkpoint helpers OK


## HTTP Helper

In [ ]:
import urllib.request
import urllib.error

# HTTP helper with retry + rate-limit backoff

def make_request(url, delay=3.0, retries=3):
    for attempt in range(retries):
        try:
            req = urllib.request.Request(
                url,
                headers={"User-Agent": "arxiv-pipeline/1.0 (research script)"},
            )
            with urllib.request.urlopen(req) as resp:
                data = resp.read()
            time.sleep(delay)
            return data
        except urllib.error.HTTPError as e:
            if e.code == 429:
                wait = delay * (2 ** attempt)
                print(f"  429 rate-limit, waiting {wait:.0f}s...")
                time.sleep(wait)
            else:
                raise
    raise Exception("Max retries exceeded")

print("HTTP helper OK")

## XML Parsing Helpers


In [ ]:
import urllib.parse

# XML parsing helpers

def _get_text(entry, tag):
    return entry.findtext(tag, default="", namespaces=NS).strip()

def _extract_authors(entry):
    return [
        a.find("atom:name", NS).text.strip() 
        for a in entry.findall("atom:author", NS)
    ]


def _extract_categories(entry):
    return [
        c.attrib.get("term") 
        for c in entry.findall("atom:category", NS) 
        if c.attrib.get("term")
    ]


def _extract_pdf_link(entry):
    return next(
        (
            l.attrib.get("href")
            for l in entry.findall("atom:link", NS)
            if l.attrib.get("title") == "pdf"
            and l.attrib.get("type") == "application/pdf"
            and l.attrib.get("rel") == "related"
        ),
        None,
    )


def _extract_primary_category(entry):
    primary = entry.find("arxiv:primary_category", NS)
    return primary.attrib.get("term") if primary is not None else None


def _parse_entry(entry):
    return {
        "arxiv_id":        _get_text(entry, "atom:id").split("/")[-1],
        "title":           _get_text(entry, "atom:title"),
        "summary":         _get_text(entry, "atom:summary"),
        "published":       _get_text(entry, "atom:published"),
        "authors":         _extract_authors(entry),
        "primary_category":_extract_primary_category(entry),
        "categories":      _extract_categories(entry),
        "pdf_link":        _extract_pdf_link(entry),
    }


def _parse_response(xml_bytes):
    root = ET.fromstring(xml_bytes)
    return [_parse_entry(e) for e in root.findall("atom:entry", NS)]

print("XML Parsing Helpers OK")

Fetch helpers OK


## arXiv fetching

In [ ]:
# Paginated fetcher

def fetch_papers(search_query=SEARCH_QUERY, max_results=MAX_RESULTS,
                 page_size=ARXIV_PAGE_SIZE, delay=REQUEST_DELAY):
    # Fetches up to max_results papers from arXiv in pages of page_size
    all_papers = []
    fetched    = 0

    while fetched < max_results:
        batch = min(page_size, max_results - fetched)
        params = urllib.parse.urlencode({
            "search_query": search_query,
            "start":        fetched,
            "max_results":  batch,
        })
        url = f"{BASE_URL}?{params}"
        print(f"  Fetching papers {fetched + 1}–{fetched + batch}...")
        xml_bytes = make_request(url, delay=delay)
        page = _parse_response(xml_bytes)
        if not page:
            print("  arXiv returned 0 entries — stopping early.")
            break
        all_papers.extend(page)
        fetched += len(page)

    print(f"Fetched {len(all_papers)} paper records.")
    return all_papers


print("Fetch helpers OK")

## PDF download & parsing


In [ ]:
def sanitize_filename(name):
    return re.sub(r'[\\/*?:"<>|]', "_", name)


def get_pdf(paper):
    # Download PDF from HF cache or arXiv, return local path.
    filename   = f"{sanitize_filename(paper['title'])}.pdf"
    local_path = os.path.join(PDF_DIR, filename)

    # 1. Already on disk
    if os.path.exists(local_path):
        return local_path

    # 2. HF cache hit
    if pdf_exists_in_hf(filename):
        try:
            download_from_hf(filename, local_path)
            print(f"  HF cache hit: {filename}")
            return local_path
        except Exception as e:
            print(f"  HF download failed for {filename}: {e}")

    # 3. Download from arXiv
    if not paper.get("pdf_link"):
        raise ValueError(f"No PDF link for {paper['arxiv_id']}")

    with _arxiv_semaphore:
        data = make_request(paper["pdf_link"], delay=1.0)
    with open(local_path, "wb") as f:
        f.write(data)

    queue_hf_upload(local_path)
    return local_path


def parse_pdf(paper, path):
    # Extract per-page text + metadata from a PDF.
    doc   = fitz.open(path)
    pages = []
    for i, page in enumerate(doc, start=1):
        raw = page.get_text("text")
        pages.append({
            "text":     raw,
            "raw_text": raw,
            "metadata": {
                "page_number":  i,
                "page_count":   len(doc),
                "format":       doc.metadata.get("format", ""),
                "creationDate": doc.metadata.get("creationDate", ""),
            },
        })
    doc.close()
    return {**paper, "pages": pages}


print("PDF helpers OK")


PDF helpers OK


## Text cleaning


In [ ]:
def clean_text(text: str) -> str:
    # General-purpose text normaliser.
    if not text:
        return ""
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    # Strip markdown / heading decorators that leak from pymupdf
    text = re.sub(r"(?m)^\s*#{1,6}\s*", " ", text)
    text = re.sub(r"(?m)^\s*[-*_~=`]{2,}\s*$", " ", text)
    text = re.sub(r"[_*`~]+", " ", text)
    text = re.sub(r"(?<!\w)#(?=\w)", "", text)
    text = text.replace("#", " ")
    # Rejoin hyphenated line-breaks
    text = re.sub(r"([A-Za-z])-\s*\n\s*([A-Za-z])", r"\1\2", text)
    # Strip standalone page numbers
    text = re.sub(r"(?im)^\s*page\s*\d+(\s*of\s*\d+)?\s*$", " ", text)
    text = re.sub(r"(?m)^\s*\d+\s*$", " ", text)
    text = re.sub(r"[—–]+", " - ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def clean_page_text(text: str, page_number=None) -> str:
    # Clean a single page; on page 1 trim everything before 'Abstract'.
    text = clean_text(text)
    if not text:
        return ""
    if page_number == 1:
        m = re.search(r"\babstract\b\s*[:-]?\s*", text, flags=re.IGNORECASE)
        if m:
            text = text[m.start():]
    return re.sub(r"\s+", " ", text).strip()


def clean_pages(paper):
    # Clean all pages and drop those that are too short.
    clean = []
    for page in paper["pages"]:
        raw     = page.get("raw_text") or page.get("text", "")
        cleaned = clean_page_text(raw, page_number=page["metadata"].get("page_number"))
        if cleaned and len(cleaned) >= MIN_PAGE_LENGTH:
            clean.append({**page, "text": cleaned, "raw_text": raw})
    return {**paper, "pages": clean}


print("Cleaning helpers OK")

Cleaning helpers OK


## Section-aware block extraction


In [ ]:
def extract_section_blocks(page_text: str):
    # Split a page into (section_heading, body_text) pairs
    pattern = re.compile(
        r'^(##.*|_?\*{0,2}Abstract\*{0,2}_?.*|_?\*{0,2}Keywords:.*\*{0,2}_?)$',
        re.IGNORECASE | re.MULTILINE,
    )
    matches = list(pattern.finditer(page_text))
    if not matches:
        return [{"section": "Unknown", "text": page_text.strip()}]

    blocks = []
    for i, match in enumerate(matches):
        section = re.sub(r'[*_`#]+', '', match.group()).strip()
        start   = match.end()
        end     = matches[i + 1].start() if i + 1 < len(matches) else len(page_text)
        body    = page_text[start:end].strip()
        if body:
            blocks.append({"section": section, "text": body})
    return blocks or [{"section": "Unknown", "text": page_text.strip()}]


def should_keep_section(section_name):
    return section_name.strip().title() not in EXCLUDED_SECTIONS


def extract_blocks(paper):
    # Turn a paper's cleaned pages into a flat list of section blocks.
    all_blocks = []

    for page in paper["pages"]:
        raw      = page.get("raw_text") or page.get("text", "")
        sections = extract_section_blocks(raw)
        page_blocks = []

        for blk in sections:
            if not should_keep_section(blk["section"]):
                continue
            body = clean_text(blk["text"])
            if not body:
                continue
            page_blocks.append({
                "section": blk["section"],
                "text":    body,
                "metadata": {
                    "arxiv_id":        paper["arxiv_id"],
                    "title":           paper["title"],
                    "authors":         ", ".join(paper.get("authors", [])),
                    "published":       paper.get("published", ""),
                    "primary_category":paper.get("primary_category", ""),
                    "categories":      ", ".join(paper.get("categories", [])),
                    "page_number":     page["metadata"]["page_number"],
                    "page_count":      page["metadata"]["page_count"],
                    "creationDate":    page["metadata"].get("creationDate", ""),
                    "format":          page["metadata"].get("format", ""),
                },
            })

        if page_blocks:
            all_blocks.extend(page_blocks)
        elif all_blocks:
            cont = clean_text(page.get("text", ""))
            if cont:
                all_blocks[-1]["text"] = f"{all_blocks[-1]['text']} {cont}".strip()

    return all_blocks


print("Block extraction helpers OK")

Block extraction helpers OK


## Chunking


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n## ", "\n### ", "\n\n", "\n", ". ", " ", ""],
)


def chunk_blocks(blocks, paper_chunk_counter):
    # Split section blocks into final chunks.
    chunks = []
    for block_i, blk in enumerate(blocks):
        if not blk["text"].strip():
            continue
        pieces = splitter.split_text(blk["text"])
        for piece_i, piece in enumerate(pieces):
            piece = re.sub(r"^[\s.,;:-]+", "", piece).strip()
            if not piece:
                continue
            chunks.append({
                "text": piece,
                "metadata": {
                    **blk["metadata"],
                    "section":         blk["section"],
                    "page_block_index": block_i,
                    "page_chunk_index": piece_i,
                    "chunk_index":      paper_chunk_counter[0],
                },
            })
            paper_chunk_counter[0] += 1

    return chunks[:MAX_CHUNKS_PER_PAPER]


print("Chunking helpers OK")

Chunking helpers OK


## NER enrichment


In [ ]:
# Load spaCy model once
nlp = spacy.load("en_core_web_sm", enable=["ner"])

SKIP_ENTITY_LABELS = {"CARDINAL", "ORDINAL"}


def _ner_text_for_chunk(chunk):
    # Prepare a sanitised, length-capped string for NER.
    text = chunk.get("text", "")
    md   = chunk.get("metadata", {})
    text = re.sub(r"[*_`~^—–]+", " ", text)
    text = re.sub(r"^[\s.,;:-]+", "", text)
    # Page 1: strip title and author lines so NER doesn't flag them as entities
    if md.get("page_number") == 1:
        title = md.get("title", "")
        if title:
            text = re.sub(re.escape(title), " ", text, flags=re.IGNORECASE)
        for author in (md.get("authors") or "").split(","):
            author = author.strip()
            if author:
                text = re.sub(rf"\b{re.escape(author)}\b", " ", text, flags=re.IGNORECASE)
        parts = re.split(r"\babstract\b[:\-\s]*", text, maxsplit=1, flags=re.IGNORECASE)
        if len(parts) == 2:
            text = parts[1]
    return re.sub(r"\s+", " ", text).strip()[:400]


def _clean_entities(raw):
    # Deduplicate and filter noisy entity types.
    seen, out = set(), []
    for text, label in raw:
        text = text.strip()
        if not text or len(text) < 3:
            continue
        if re.fullmatch(r"\d+(\.\d+)?", text):
            continue
        if label in SKIP_ENTITY_LABELS:
            continue
        if re.fullmatch(r"[^\w]+", text):
            continue
        key = (text.lower(), label)
        if key not in seen:
            seen.add(key)
            out.append((text, label))
    return out


def enrich_with_ner(chunks):
    # Run spaCy NER over all chunks and add entity fields to metadata.
    texts = [
        _ner_text_for_chunk(c) if len(c.get("text", "")) > 80 else ""
        for c in chunks
    ]
    # Batch NER through spaCy's pipe for efficiency
    docs = list(nlp.pipe(texts))
    for chunk, doc, ner_text in zip(chunks, docs, texts):
        raw     = [(ent.text, ent.label_) for ent in doc.ents]
        cleaned = _clean_entities(raw)
        chunk["metadata"]["entities"]     = ", ".join(t for t, _ in cleaned)
        chunk["metadata"]["entity_labels"]= ", ".join(l for _, l in cleaned)
        chunk["metadata"]["ner_text"]     = ner_text
    return chunks


print("NER helpers OK")

NER helpers OK


## Embedding & ChromaDB storage


In [ ]:
# ChromaDB setup
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

try:
    chroma_client.delete_collection(COLLECTION_NAME)
    print(f"Deleted existing collection '{COLLECTION_NAME}'")
except:
    print(f"No existing collection named '{COLLECTION_NAME}'")

col = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    configuration={"hnsw": {"space": "cosine"}},
)
print("ChromaDB collection ready.")


# Embedding
def get_embeddings(texts):
    resp = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [item.embedding for item in resp.data]


def embed_chunks(chunks):
    # Embed all chunks in sequential batches with exponential-backoff on rate limits.
    texts      = [c["text"] for c in chunks]
    embeddings = []

    for i in range(0, len(texts), EMBED_BATCH_SIZE):
        batch = texts[i:i + EMBED_BATCH_SIZE]
        for attempt in range(6):
            try:
                embeddings.extend(get_embeddings(batch))
                break
            except RateLimitError:
                wait = 2 ** attempt
                print(f"  Rate-limit hit, retrying in {wait}s...")
                time.sleep(wait)

    return embeddings


# Storage 
METADATA_KEYS = [
    "arxiv_id", "title", "section", "primary_category", "categories",
    "authors", "published", "creationDate", "page_number", "page_count",
    "format", "page_block_index", "page_chunk_index", "chunk_index",
    "entities", "entity_labels",
]


def store_chunks(chunks, embeddings):
    # Upsert chunks into ChromaDB in batches to avoid memory issues at scale.
    ids   = [f"{c['metadata']['arxiv_id']}_chunk_{c['metadata']['chunk_index']}" for c in chunks]
    docs  = [c["text"] for c in chunks]
    metas = [
        {k: str(c["metadata"].get(k, "")) for k in METADATA_KEYS}
        for c in chunks
    ]

    total = len(docs)
    for i in range(0, total, CHROMA_UPSERT_BATCH):
        col.add(
            ids=ids[i:i + CHROMA_UPSERT_BATCH],
            documents=docs[i:i + CHROMA_UPSERT_BATCH],
            metadatas=metas[i:i + CHROMA_UPSERT_BATCH],
            embeddings=embeddings[i:i + CHROMA_UPSERT_BATCH],
        )
    print(f"  Stored {total} chunks in ChromaDB.")


print("Embedding & storage helpers OK")

Deleted existing collection 'papers'
ChromaDB collection ready.
Embedding & storage helpers OK


## Main pipeline


In [ ]:
def process_paper(paper, processed_ids, global_chunk_counter):
    # Full pipeline for a single paper:
    # download → parse → clean → block extraction → chunking → NER → embed → store
    if paper["arxiv_id"] in processed_ids:
        print(f"  Skipping {paper['arxiv_id']} (already processed)")
        return

    try:
        # 1. Download PDF
        path = get_pdf(paper)

        # 2. Parse pages
        parsed = parse_pdf(paper, path)

        # 3. Clean pages
        cleaned = clean_pages(parsed)
        if not cleaned["pages"]:
            print(f"  No usable pages for {paper['arxiv_id']}, skipping.")
            return

        # 4. Extract section blocks
        blocks = extract_blocks(cleaned)
        if not blocks:
            print(f"  No blocks extracted for {paper['arxiv_id']}, skipping.")
            return

        # 5. Chunk
        chunks = chunk_blocks(blocks, global_chunk_counter)
        if not chunks:
            return

        # 6. NER enrichment
        chunks = enrich_with_ner(chunks)

        # 7. Embed
        embeddings = embed_chunks(chunks)

        # 8. Store
        store_chunks(chunks, embeddings)

        # 9. Checkpoint + cleanup
        processed_ids.add(paper["arxiv_id"])
        save_checkpoint(processed_ids)
        os.remove(path)

        del parsed, cleaned, blocks, chunks, embeddings
        gc.collect()

        print(f"  Done: {paper['arxiv_id']} — '{paper['title'][:60]}'")

    except Exception as e:
        logging.error(f"FAIL {paper['arxiv_id']}: {e}")
        print(f"  FAILED {paper['arxiv_id']}: {e}")


def run():
    processed_ids      = load_checkpoint()
    global_chunk_counter = [0]         

    print("Fetching paper metadata...")
    papers = fetch_papers()

    remaining = [p for p in papers if p["arxiv_id"] not in processed_ids]
    print(f"{len(remaining)} papers to process ({len(processed_ids)} already in checkpoint).")

    for i, paper in enumerate(remaining, start=1):
        print(f"\n[{i}/{len(remaining)}] {paper['arxiv_id']}")
        process_paper(paper, processed_ids, global_chunk_counter)

    flush_hf_uploads()
    print(f"\nDONE — processed {len(processed_ids)} papers, {global_chunk_counter[0]} total chunks stored.")


print("Pipeline OK — call run() to start.")

Pipeline OK — call run() to start.


In [13]:
run()

Fetching paper metadata...
  Fetching papers 1–100...
  Fetching papers 101–200...
  Fetching papers 201–300...
  Fetching papers 301–400...
  Fetching papers 401–500...
  Fetching papers 501–600...
  Fetching papers 601–700...
  Fetching papers 701–800...
  Fetching papers 801–900...
  Fetching papers 901–1000...
Fetched 1000 paper records.
1000 papers to process (0 already in checkpoint).

[1/1000] 1002.2191v1
HF repo contains 199 files.
  HF cache hit: Vision Based Game Development Using Human Computer Interaction.pdf
  Stored 36 chunks in ChromaDB.
  Done: 1002.2191v1 — 'Vision Based Game Development Using Human Computer Interacti'

[2/1000] 0407054v2
  HF cache hit: From truth to computability I.pdf
  Stored 245 chunks in ChromaDB.
  Done: 0407054v2 — 'From truth to computability I'

[3/1000] 1002.2409v1
  HF cache hit: Changing Neighbors k Secure Sum Protocol for Secure Multi Party Computation.pdf
  Stored 21 chunks in ChromaDB.
  Done: 1002.2409v1 — 'Changing Neighbors k Secure 

KeyboardInterrupt: 

## Retrieval testing


In [ ]:
def retrieve_chunks(query, n_results=5, filter_category=None):
    # Embed query and return nearest chunks from ChromaDB.
    query_embedding = get_embeddings([query])[0]
    where = {"primary_category": filter_category} if filter_category else None
    return col.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        where=where,
    )


def _compute_entity_overlap(entities_str, query):
    entity_set  = {e.strip().lower() for e in entities_str.split(",") if e.strip()}
    query_terms = {t.lower().strip() for t in query.split() if t.strip()}
    return len(entity_set & query_terms)


def semantic_search(query, n_results=5, min_similarity=0.25,
                    filter_category=None, use_hybrid=False, entity_bonus=0.5):
    # Search ChromaDB and return formatted results.
    # use_hybrid=True: re-ranks by cosine_similarity + entity_overlap bonus.
    raw       = retrieve_chunks(query, n_results=n_results, filter_category=filter_category)
    documents = raw.get("documents", [[]])[0]
    metadatas = raw.get("metadatas", [[]])[0]
    ids       = raw.get("ids",       [[]])[0]
    distances = raw.get("distances", [[]])[0] if "distances" in raw else []

    results = []
    for i, doc_text in enumerate(documents):
        meta     = metadatas[i] if i < len(metadatas) else {}
        chunk_id = ids[i]       if i < len(ids)       else f"chunk_{i}"
        cos_dist = distances[i] if i < len(distances)  else None
        cos_sim  = (1 - cos_dist) if cos_dist is not None else None

        if cos_sim is not None and cos_sim < min_similarity:
            continue

        entities       = meta.get("entities", "")
        entity_overlap = _compute_entity_overlap(entities, query)
        hybrid_score   = (
            (cos_sim + entity_overlap * entity_bonus) if use_hybrid and cos_sim is not None
            else cos_sim
        )

        results.append({
            "rank":             i + 1,
            "chunk_id":         chunk_id,
            "cosine_similarity":cos_sim,
            "hybrid_score":     hybrid_score,
            "entity_overlap":   entity_overlap,
            "text":             doc_text,
            "arxiv_id":         meta.get("arxiv_id", ""),
            "title":            meta.get("title", ""),
            "section":          meta.get("section", ""),
            "primary_category": meta.get("primary_category", ""),
            "authors":          meta.get("authors", ""),
            "published":        meta.get("published", ""),
            "page_number":      meta.get("page_number", ""),
            "chunk_index":      meta.get("chunk_index", ""),
            "entities":         entities,
            "entity_labels":    meta.get("entity_labels", ""),
        })

    if use_hybrid:
        results.sort(key=lambda x: x["hybrid_score"] or -1, reverse=True)
        for idx, r in enumerate(results, start=1):
            r["rank"] = idx

    return results


def display_search_results(results, query=""):
    sep = "=" * 60
    if query:
        print(f"\n{sep}\nQuery: {query!r}\n{sep}")
    for r in results:
        print(f"\nResult #{r['rank']}  |  chunk: {r['chunk_id']}")
        print(f"  Paper  : {r['title'][:70]}")
        print(f"  Authors: {r['authors'][:60]}")
        print(f"  Section: {r['section']}  |  Page: {r['page_number']}")
        print(f"  Cosine similarity : {r['cosine_similarity']:.4f}" if r['cosine_similarity'] else "  Cosine: N/A")
        print(f"  Entities          : {r['entities'][:80]}")
        print(f"  Text preview      : {r['text'][:200]}")


# Quick smoke-test queries
TEST_QUERIES = [
    "What are the main challenges in large language model training?",
    "How does attention mechanism work in transformers?",
    "What datasets are commonly used for computer vision benchmarks?",
]

for q in TEST_QUERIES:
    results = semantic_search(q, n_results=3)
    display_search_results(results, query=q)

## RAG pipeline


In [ ]:
RAG_SYSTEM_PROMPT = """
You are an academic research assistant for an enterprise knowledge mining system built on arXiv papers.

Instructions:
- Answer questions only from the retrieved context.
- Treat the retrieved text as excerpts from research papers.
- Prefer precise academic wording.
- Summarise contributions, methods, datasets, results, or limitations only if they are supported by the context.
- If the answer is incomplete or missing from the context, say that explicitly.
- Do not include source tags or citation brackets such as [Source 1] in the answer text.
- Write only the answer, formatted concisely and clearly.
""".strip()


def generate_answer(query, context, max_tokens=300, temperature=0.2):
    resp = client.chat.completions.create(
        model=RAG_MODEL,
        messages=[
            {"role": "system", "content": RAG_SYSTEM_PROMPT},
            {
                "role": "user",
                "content": (
                    f"Use the retrieved research-paper context below to answer the question.\n\n"
                    f"Context:\n{context}\n\n"
                    f"Question: {query}\n\nAnswer:"
                ),
            },
        ],
        max_tokens=max_tokens,
        temperature=temperature,
    )
    return resp.choices[0].message.content.strip()


def rag_query(query, n_results=3, min_similarity=0.25,
              filter_category=None, use_hybrid=False, entity_bonus=0.5):
    results = semantic_search(
        query=query, n_results=n_results, min_similarity=min_similarity,
        filter_category=filter_category, use_hybrid=use_hybrid, entity_bonus=entity_bonus,
    )
    if not results:
        return {
            "query":   query,
            "answer":  "No relevant context found in the knowledge base.",
            "sources": [],
        }

    context_parts = []
    sources       = []
    for i, r in enumerate(results):
        context_parts.append(
            f"[Source {i+1}: {r['title']}, Section: {r['section']}, "
            f"Page {r['page_number']}, Chunk {r['chunk_index']}]\n{r['text']}"
        )
        sources.append({
            "source_number":    i + 1,
            "chunk_id":         r["chunk_id"],
            "arxiv_id":         r["arxiv_id"],
            "title":            r["title"],
            "section":          r["section"],
            "primary_category": r["primary_category"],
            "page_number":      r["page_number"],
            "chunk_index":      r["chunk_index"],
            "cosine_similarity":r["cosine_similarity"],
            "hybrid_score":     r["hybrid_score"],
            "entity_overlap":   r["entity_overlap"],
        })

    context = "\n\n".join(context_parts)
    answer  = generate_answer(query, context)
    return {"query": query, "answer": answer, "sources": sources}


def display_rag_response(response):
    sep = "=" * 60
    print(f"\n{sep}")
    print(f"Query  : {response['query']}")
    print(f"\nAnswer :\n{response['answer']}")
    print("\nSources:")
    for s in response["sources"]:
        sim = f"{s['cosine_similarity']:.4f}" if s['cosine_similarity'] else "N/A"
        print(f"  [{s['source_number']}] {s['title'][:60]} | section: {s['section']} | sim: {sim}")
    print(sep)


# RAG test queries
RAG_TEST_QUERIES = [
    "What are the main challenges in large language model training?",
    "How does attention mechanism work in transformers?",
]

for q in RAG_TEST_QUERIES:
    resp = rag_query(q, n_results=3)
    display_rag_response(resp)